# CEG-WM runtime qualification

Scope: real SD3.5 runtime qualification only. Start with `smoke`; do not use this notebook for calibration, attacks, LF/routing promotion, experiments, or stage migration.

In [ ]:
PROFILE = "smoke"
REPLAY_SOURCE = None
DRIVE_ROOT = "/content/drive/MyDrive/CEG-WM/runtime_qualification"
PACKAGE_ZIP = f"{DRIVE_ROOT}/execution_packages/current/ceg_wm_runtime_execution.zip"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, shutil, subprocess, torch
assert torch.cuda.is_available(), "GPU_REQUIRED: CUDA is unavailable"
print(torch.cuda.get_device_name(0))
print(shutil.disk_usage("/content"))

In [ ]:
from pathlib import Path
TEMP_ROOT = Path("/content/ceg_wm_runtime")
PACKAGE_ROOT = TEMP_ROOT / "package"
PIP_CACHE = Path("/content/pip_cache")
HF_CACHE = Path("/content/hf_cache")
for path in (TEMP_ROOT, PACKAGE_ROOT, PIP_CACHE, HF_CACHE): path.mkdir(parents=True, exist_ok=True)
assert not any(PACKAGE_ROOT.iterdir()), "ephemeral package directory must start empty"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["HF_HOME"] = str(HF_CACHE)

In [ ]:
import hashlib, json, re, zipfile
from pathlib import PurePosixPath

PACKAGE_REQUIRED_FILES = {
    "README.md",
    "configs/runtime/runtime_sd35_flowmatch.json",
    "pyproject.toml",
    "requirements_runtime_qualification.txt",
    "scripts/experiment_execution/__init__.py",
    "scripts/experiment_execution/runtime_qualification_runner.py",
}
PACKAGE_INCLUDE_ROOTS = ("main/", "runtime/")
PACKAGE_EXCLUDED_PARTS = {".agents", ".codex", ".git", ".pytest_cache", "__pycache__", "governance", "notebooks", "outputs"}
SENSITIVE_PARTS = (".env", "credential", "secret", "private_key", "id_rsa", "id_ed25519")

def safe_extract_package(archive_path, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive_path) as archive:
        members = archive.infolist()
        names = [member.filename for member in members]
        assert len(names) == len(set(names)), "duplicate zip member"
        assert "runtime_execution_manifest.json" in names, "package manifest missing"
        assert sum(member.file_size for member in members) <= 64 * 1024 * 1024, "package is unexpectedly large"
        for member in members:
            name = member.filename
            path = PurePosixPath(name)
            assert name and "\\" not in name and "\x00" not in name, "unsafe zip path"
            assert not path.is_absolute() and ".." not in path.parts and not re.match(r"^[A-Za-z]:", name), "unsafe zip path"
            assert ((member.external_attr >> 16) & 0o170000) != 0o120000, "zip symlink forbidden"
            target = (destination / Path(*path.parts)).resolve()
            assert target == destination or destination in target.parents, "unsafe zip target"
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open("xb") as sink:
                    shutil.copyfileobj(source, sink)

def verify_unpacked_package(package_root):
    package_root = Path(package_root).resolve()
    manifest_path = package_root / "runtime_execution_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    assert set(manifest) == {"copied_files", "excluded_parts", "package_ready", "package_schema_version", "profile_name", "runtime_candidate_revision"}, "manifest schema drifted"
    revision = manifest["runtime_candidate_revision"]
    assert manifest["package_schema_version"] == 1, "manifest schema version drifted"
    assert manifest["profile_name"] == "experiment_execution_package", "manifest profile drifted"
    assert manifest["package_ready"] is True, "package is not ready"
    assert isinstance(revision, str) and re.fullmatch(r"[0-9a-f]{40}", revision), "runtime candidate revision is invalid"
    assert manifest["excluded_parts"] == sorted(PACKAGE_EXCLUDED_PARTS), "manifest exclusions drifted"
    entries = manifest["copied_files"]
    assert isinstance(entries, list) and entries, "manifest file list is invalid"
    expected = {}
    for entry in entries:
        assert isinstance(entry, dict) and set(entry) == {"path", "sha256", "size_bytes"}, "manifest entry is invalid"
        path_text, digest, size = entry["path"], entry["sha256"], entry["size_bytes"]
        assert isinstance(path_text, str) and path_text, "manifest path is invalid"
        assert isinstance(digest, str) and re.fullmatch(r"[0-9a-f]{64}", digest), "manifest digest is invalid"
        assert type(size) is int and size >= 0, "manifest size is invalid"
        path = PurePosixPath(path_text)
        assert "\\\\" not in path_text and "\x00" not in path_text and not re.match(r"^[A-Za-z]:", path_text), "unsafe manifest path"
        assert not path.is_absolute() and ".." not in path.parts, "unsafe manifest path"
        assert not any(part in PACKAGE_EXCLUDED_PARTS for part in path.parts), "excluded manifest path"
        assert not any(marker in part.lower() for part in path.parts for marker in SENSITIVE_PARTS), "sensitive manifest path"
        assert path_text in PACKAGE_REQUIRED_FILES or path_text.startswith(PACKAGE_INCLUDE_ROOTS), "unallowlisted manifest path"
        assert path_text not in expected, "duplicate manifest path"
        expected[path_text] = (size, digest)
    actual = {}
    for candidate in package_root.rglob("*"):
        assert not candidate.is_symlink(), "package symlink forbidden"
        if candidate.is_file():
            relative = candidate.relative_to(package_root).as_posix()
            if relative != "runtime_execution_manifest.json":
                actual[relative] = candidate
    assert set(actual) == set(expected), "package file set differs from manifest"
    assert PACKAGE_REQUIRED_FILES <= set(expected), "required package file missing"
    assert any(path.startswith("main/") for path in expected) and any(path.startswith("runtime/") for path in expected), "method or runtime package root missing"
    for path_text, candidate in actual.items():
        size, digest = expected[path_text]
        payload = candidate.read_bytes()
        assert len(payload) == size and hashlib.sha256(payload).hexdigest() == digest, f"package file identity drifted: {path_text}"
    return manifest

safe_extract_package(PACKAGE_ZIP, PACKAGE_ROOT)
manifest = verify_unpacked_package(PACKAGE_ROOT)
REVISION = manifest["runtime_candidate_revision"]

In [ ]:
subprocess.run(["python", "-m", "pip", "install", "--cache-dir", str(PIP_CACHE), "-r", str(PACKAGE_ROOT / "requirements_runtime_qualification.txt")], check=True)

In [ ]:
from google.colab import userdata
HF_TOKEN_VALUE = userdata.get("HF_TOKEN")
ROOT_KEY_VALUE = userdata.get("CEG_WM_ROOT_KEY")
assert isinstance(HF_TOKEN_VALUE, str) and HF_TOKEN_VALUE, "HF_TOKEN Secret is required"
assert isinstance(ROOT_KEY_VALUE, str) and ROOT_KEY_VALUE, "CEG_WM_ROOT_KEY Secret is required"
os.environ["HF_TOKEN"] = HF_TOKEN_VALUE
os.environ["CEG_WM_ROOT_KEY"] = ROOT_KEY_VALUE

In [ ]:
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RESULT_DIR = Path(DRIVE_ROOT) / "runs" / REVISION / RUN_ID
RESULT_DIR.mkdir(parents=True, exist_ok=False)
RESULT_ZIP = RESULT_DIR / f"ceg_wm_runtime_qualification_{RUN_ID}.zip"
TEMP_RESULT_ZIP = TEMP_ROOT / f"ceg_wm_runtime_qualification_{RUN_ID}.zip"
command = ["python", "-m", "scripts.experiment_execution.runtime_qualification_runner", "--profile", PROFILE, "--run-id", RUN_ID, "--package-root", str(PACKAGE_ROOT), "--runtime-candidate-revision", REVISION, "--result-zip", str(TEMP_RESULT_ZIP), "--ephemeral-root", str(TEMP_ROOT), "--persistent-root", DRIVE_ROOT]
if PROFILE == "replay":
    assert REPLAY_SOURCE, "replay profile requires an existing qualification result zip"
    command.extend(["--replay-source", REPLAY_SOURCE])
completed = subprocess.run(command, cwd=PACKAGE_ROOT, text=True, capture_output=True)
print(completed.stdout)
print(completed.stderr)
assert TEMP_RESULT_ZIP.exists(), "runner failed without a result zip"
shutil.copy2(TEMP_RESULT_ZIP, RESULT_ZIP)
assert completed.returncode in (0, 1, 2), f"unexpected runner exit code {completed.returncode}; result zip saved to {RESULT_ZIP}"
with zipfile.ZipFile(TEMP_RESULT_ZIP) as archive:
    assert set(archive.namelist()) == {"run_summary.json", "environment_summary.json", "runtime_checks.jsonl", "failures.jsonl"}, "result file set drifted"
    summary = json.loads(archive.read("run_summary.json"))
assert summary["result_schema_version"] == 2, "result schema version drifted"
assert summary["run_id"] == RUN_ID and summary["profile"] == PROFILE and summary["runtime_candidate_revision"] == REVISION and summary["result_zip_filename"] == TEMP_RESULT_ZIP.name, "result identity drifted"
assert (completed.returncode == 0) == (summary["run_status"] == "passed"), "runner exit/status drifted"
if completed.returncode != 0:
    raise RuntimeError(f"runner failed with exit code {completed.returncode}; failure zip saved to {RESULT_ZIP}")

In [ ]:
with zipfile.ZipFile(RESULT_ZIP) as archive:
    summary = json.loads(archive.read("run_summary.json"))
print({"result_zip": str(RESULT_ZIP), "profile": summary["profile"], "run_status": summary["run_status"], "failures": summary["failure_count"]})